In [1]:
from datetime import datetime, timedelta, timezone
import gsw
import pandas as pd
import xarray as xr

from cms import CMS

In [2]:
edt = datetime.now(timezone.utc).replace(tzinfo=None)
bdt = edt - timedelta(days = 1)

In [ ]:
hatcms = CMS(verbose = True)
active_insts = hatcms.get_instruments(begin_datetime = bdt, end_datetime = edt)

idss = []
for active_inst in active_insts[::-1]:
    inst_id = active_inst['sensor_id']
    inst_vars = hatcms.get_instrument_variables(inst_id, begin_datetime = bdt, end_datetime = edt)

    vdss = []
    for inst_var in inst_vars:
        var_name = inst_var['detailed_name']
        data_model = inst_var['data_model']
        data_field = inst_var['data_fieldname']
        var_data = hatcms.get_variable_data(data_model, begin_datetime=bdt, end_datetime=edt)
        var_data = [{k:v for k,v in vd.items() if k in ['time', data_field]} for vd in var_data]
        vdf = pd.DataFrame(var_data)
        vdf['time'] = pd.to_datetime(vdf['time'])
        vdf.index = vdf.time.dt.tz_localize(None)
        vdf = vdf.drop(columns = 'time', errors='ignore')
        vdf = vdf.rename(columns = {data_field: var_name})
        vds = vdf.to_xarray()

        # Variable level attributes.
        valid_min = inst_var['gross_min'] if not None else inst_var['global_min']
        valid_max = inst_var['gross_max'] if not None else inst_var['global_max']
        vds[var_name].attrs['short_name'] = inst_var['short_name']
        vds[var_name].attrs['long_name'] = inst_var['long_name']
        vds[var_name].attrs['comment'] = inst_var['description']
        vds[var_name].attrs['valid_min'] = valid_min
        vds[var_name].attrs['valid_max'] = valid_max
        vds[var_name].attrs['resolution'] = inst_var['resolution']
        vds[var_name].attrs['units'] = inst_var['units_abbrev']
        vds[var_name].attrs['sensor_id'] = inst_var['sensor_id']

        vdss.append(vds[var_name])
    ids = xr.combine_by_coords(vdss, data_vars = 'all', join = 'outer', combine_attrs = 'drop_conflicts')
    idss.append(ids)

ds = xr.combine_by_coords(idss, data_vars = 'all', join = 'outer', combine_attrs = 'drop_conflicts')

# Drop raw level variables or variables that don't matter to the typical science user.
drop_vars = ['absorbance_254_raw_unitless',
             'absorbance_350_raw_unitless',
             'asm_raw_counts',
             'b700_raw_counts',
             'battery_voltage_raw_v',
             'chlorophyll_concentration_in_sea_water_raw_counts',
             'chlorophyll_raw_rfu',
             'dark_value_raw_counts',
             'external_power_voltage_raw_v',
             'fdom_raw_counts',
             'internal_relative_humidity_raw_percent',
             'internal_temperature_raw_degrees_c',
             'internal_voltage_raw_v',
             'lamp_temp_raw_degrees_c',
             'lamp_voltage_raw_v',
             'main_current_raw_ma',
             'main_voltage_raw_v',
             'pe_raw_rfu',
             'spectrometer_temperature_raw_degrees_c',
             'cumulative_lamp_time_raw_s',
             'bromide_internally_derived_mg_per_l',
             'depth_internally_derived_m',
             'nitrogen_in_nitrate_processed_corrected_mg_n_per_l']
ds = ds.drop_vars(drop_vars, errors='ignore')

name_mapper = {'beta700_processed_per_m_per_sr': 'beta_700',
               'chlorophyll_concentration_in_sea_water_processed_ug_per_l': 'chlorophyll_a',
               'fdom_processed_ppb': 'fdom',
               'nitrate_concentration_processed_corrected_umol': 'nitrate_concentration',
               'ph_seawater_internally_derived_unitless': 'sea_water_ph',
               'pressure_raw_psi_a': 'sea_water_pressure',
               'sea_water_practical_salinity_internally_derived_unitless': 'sea_water_practical_salinity',
               'sea_water_specific_conductance_internally_derived_ms_per_cm': 'sea_water_electrical_conductivity',
               'sea_water_temperature_raw_degrees_c': 'sea_water_temperature',
               'sea_water_turbidity_raw_fnu':'turbidity',
               'dissolved_oxygen_internally_derived_mg_per_l':'dissolved_oxygen',
               'dissolved_oxygen_sat_internally_derived_percent': 'oxygen_saturation'}
ds = ds.rename({k:v for k,v in name_mapper.items() if k in ds.data_vars})


# Convert data.
lat =  44.625413
lon = -124.044851

dvs = ds.data_vars
if 'sea_water_pressure' in dvs:
    if ds['sea_water_pressure'].attrs['units'] == 'psi a':
        ds['sea_water_pressure'] = ds['sea_water_pressure'] * 0.689476
        ds['sea_water_pressure'].attrs['units'] = 'decibar'
        ds['sea_water_pressure'].attrs['valid_min'] = ds['sea_water_pressure'].attrs['valid_min'] * 0.689476
        ds['sea_water_pressure'].attrs['valid_max'] = ds['sea_water_pressure'].attrs['valid_max'] * 0.689476

if 'sea_water_practical_salinity' in dvs and 'sea_water_temperature' in dvs and 'sea_water_pressure' in dvs:
    sp = ds.sea_water_practical_salinity
    t = ds.sea_water_temperature
    p = ds.sea_water_pressure
    sa = gsw.SA_from_SP(sp, p, lon, lat)
    ct = gsw.CT_from_t(sa, t, p)
    rho = gsw.rho(sa, ct, p)
    ds['sea_water_density'] = rho

if 'dissolved_oxygen' in dvs and 'sea_water_practical_salinity' in dvs and 'sea_water_temperature' in dvs and 'sea_water_pressure' in dvs:
    if ds.dissolved_oxygen.attrs['units'] == 'mg/l':
        pref = 0
        do = ds.dissolved_oxygen * 22.391/31.998 # Convert mg/l to ml/l.
        sp = ds.sea_water_practical_salinity.interp({'time': ds.time.values}, method = 'nearest', kwargs = {'fill_value': 'extrapolate'})
        t = ds.sea_water_temperature.interp({'time': ds.time.values}, method = 'nearest', kwargs = {'fill_value': 'extrapolate'})
        p = ds.sea_water_pressure.interp({'time': ds.time.values}, method = 'nearest', kwargs = {'fill_value': 'extrapolate'})
        sa = gsw.SA_from_SP(sp, p, lon, lat)
        ct = gsw.CT_from_t(sa, t, p)

        pot_rho = gsw.pot_rho_t_exact(sa,t,p, pref)
        do = do * 44661 / pot_rho # Convert ml/l to umol/kg.

        ds['dissolved_oxygen'] = do
        ds['dissolved_oxygen'].attrs['units'] = 'umol kg-1'
        ds['dissolved_oxygen'].attrs['valid_min'] = 0
        ds['dissolved_oxygen'].attrs['valid_min'] = 500

        oxysol = gsw.O2sol(sa, ct, p, lon, lat)
        oxy_sat = ds['dissolved_oxygen']/oxysol * 100
        ds['oxygen_saturation'] = oxy_sat
        ds['oxygen_saturation'].attrs['units'] = '%'
        ds['oxygen_saturation'].attrs['valid_min'] = 0
        ds['oxygen_saturation'].attrs['valid_max'] = 200
        ds['oxygen_saturation'].attrs['short_name'] = 'oxy_sat'
        ds['oxygen_saturation'].attrs['short_name'] = 'Oxygen Saturation in Reference to Equilibrium with Atmosphere'
        ds['oxygen_saturation'].attrs['comment'] = 'Oxygen saturation in percent.'
else:
    ds = ds.drop_vars(['dissolved_oxygen'], errors='ignore')

In [ ]:
ds

In [ ]:
ds.sea_water_pressure.dropna(dim = 'time').plot()

In [ ]:
ds.sea_water_practical_salinity.dropna(dim ='time').plot()

In [ ]:
ds.dissolved_oxygen.dropna(dim = 'time').plot()